# Repaso — Semana 6

[Índice de la semana](semana-06.ipynb) | [Anterior: Unidad 40](clase-2-mini-proyectos/u40-pandemia-coronavirus.ipynb)

Todo lo de la semana en una página: mapa de temas, cheatsheet por unidad y errores comunes.

## Mapa rápido de la semana

| Tarea | Herramienta en Pandas | Ejemplo |
|---|---|---|
| Detectar faltantes | `isnull()` / `isna()` | `df.isnull().sum()` |
| Eliminar faltantes | `dropna()` | `df.dropna(subset=['col'])` |
| Reemplazar faltantes | `fillna()` | `df['col'] = df['col'].fillna(df['col'].mean())` |
| Resumen estadístico | `describe()`, `mean()`, `median()`, `mode()`, `var()`, `std()` | `df.describe()` |
| Fechas | `pd.to_datetime()`, `pd.Timestamp`, `pd.Period` | `df.index = pd.to_datetime(df['fecha'])` |
| Ventana móvil | `rolling(n).mean()` | `s.rolling(20).mean()` |
| Cambio porcentual | `pct_change()` | `precios.pct_change() * 100` |
| Rendimiento acumulado | `cumprod()` | `(1 + precios.pct_change()).cumprod() - 1` |
| Correlación | `corr()` | `rendimientos.corr()` |
| Mapa coroplético | `folium.Choropleth` | capa coloreada por país sobre `folium.Map` |

**Clave:** `df[['col']]` devuelve un DataFrame y `df['col']` una Series. `describe()` no incluye la moda: se obtiene con `mode()`.

## Unidad 37 — Ordenación de datos

[Ir a la unidad](clase-1-ordenacion-y-series-temporales/u37-ordenacion-de-datos.ipynb)

```python
df = sns.load_dataset('titanic', cache=True)  # carga un conjunto de datos de seaborn
df.info()  # no nulos por columna
df.isnull().sum()  # NaN por columna
df.isnull().sum().sum()  # total de NaN
df['deck'].value_counts(dropna=False)  # conteo incluyendo NaN
df.dropna(axis=1, thresh=500)  # quita columnas con menos de 500 valores válidos
df.dropna(subset=['age'])  # quita filas sin edad
df['age'] = df['age'].fillna(df['age'].mean())  # rellena con la media
df['col'] = df['col'].fillna(df['col'].mode()[0])  # rellena con la moda
df['col'].ffill()  # rellena con el valor anterior (bfill(): el posterior)
pd.read_csv(ruta, sep=';')  # CSV separado por punto y coma
df.describe()  # resumen estadístico
s.median(), s.mode(), s.var(), s.std()  # mediana, moda, varianza, desviación
df.std(numeric_only=True) / df.mean(numeric_only=True)  # coeficiente de variación
df.cov(numeric_only=True), df.corr(numeric_only=True)  # covarianza y correlación
plt.hist(s); plt.boxplot(s); plt.plot(x, y, 'o')  # histograma, caja, dispersión
sns.pairplot(df[['G1', 'G3']])  # histogramas y dispersión de varias variables
```

**Errores comunes**

- Buscar NaN con `== np.nan`: nunca es verdadero; hay que usar `isnull()` o `isna()`.
- Usar `df['col'].fillna(valor, inplace=True)`: en pandas actual modifica una copia; hay que asignar `df['col'] = df['col'].fillna(valor)`.
- Usar `fillna(method='ffill')`: el parámetro `method` fue eliminado en pandas 3; se usan `ffill()` y `bfill()`.
- Creer que `thresh` es el máximo de NaN permitidos: es el mínimo de valores no nulos que debe tener la fila o columna.
- Leer un CSV separado por `;` sin `sep=';'`, que deja todos los datos en una sola columna.
- Llamar a `std()` o `mean()` sobre un DataFrame con columnas de texto sin `numeric_only=True`.

## Unidad 38 — Datos de series temporales

[Ir a la unidad](clase-1-ordenacion-y-series-temporales/u38-series-temporales.ipynb)

```python
pd.Timestamp('2017-05-03 12:00')  # crea un punto en el tiempo (datetime64)
pd.Timedelta(days=1, hours=10)  # intervalo de tiempo
fecha + pd.Timedelta(days=1)  # suma un intervalo a una fecha
pd.Period('1973-07', freq='M')  # periodo mensual con inicio y fin
periodo.start_time, periodo.end_time  # primer y último instante del periodo
periodo + 2  # desplaza el periodo dos unidades de su frecuencia
df['N_FECHA'] = pd.to_datetime(df['FECHA'])  # texto a datetime64
pd.to_datetime(serie, errors='coerce')  # lo no convertible pasa a NaT
df = df.set_index('N_FECHA').sort_index()  # DatetimeIndex ordenado
df['2016-11-04':'2016-11-10']  # troceo por fechas, extremos incluidos
pd.date_range('2021-01-01', periods=5, freq='ME')  # marcas con frecuencia (pandas 3: ME, QE, YE, h)
pd.period_range('2020-01', '2020-12', freq='M')  # PeriodIndex mensual
serie.resample('ME').mean()  # cambia la frecuencia: promedio mensual
serie.rolling(window=5).mean()  # media móvil de 5 observaciones
df.plot(kind='area', alpha=0.2, stacked=False)  # gráfico de áreas superpuestas
```

**Errores comunes**

- Dejar la fecha como texto: no se pueden trocear rangos, remuestrear ni calcular diferencias de tiempo hasta convertirla con `to_datetime` y usarla como índice.
- Convertir directamente un año entero con `to_datetime`: se interpreta como nanosegundos desde 1970; hay que pasar antes `astype(str)` y usar `format='%Y'`.
- Usar en pandas 3 los alias antiguos `M`, `Q`, `A`, `H`: ahora son `ME`, `QE`, `YE`, `h`; y `Ms` no es inicio de mes (es `MS`).
- Olvidar que las primeras N-1 posiciones de una media móvil de ventana N son NaN.
- Sumar todas las entidades de un archivo de países sin revisar que incluye agregados (`World`, regiones, grupos de ingresos) y obtener un total mundial inflado.
- Sustituir NaN por 0 sin considerar que distorsiona promedios (solo es neutro para sumas).

## Unidad 39 — Mini proyecto de análisis de datos financieros

[Ir a la unidad](clase-2-mini-proyectos/u39-analisis-datos-financieros.ipynb)

```python
yf.Ticker("SPY").history(start=inicio, end=fin)  # descarga datos de Yahoo Finanzas (requiere conexión)
data.insert(0, "Ticker", ticker)  # agrega la columna Ticker en la posición 0
d.pivot(columns="Ticker", values="Close")  # formato ancho: una columna por ticker
pd.concat([SPY, IYW, VT], axis=1, join='outer')  # une marcos lado a lado conservando todas las fechas
stock.plot(figsize=(20, 10))  # línea por columna con las fechas en el eje horizontal
covid = stock['2020-2-1':'2020-7-31']  # corte por fechas (incluye ambos extremos)
plt.bar(df.index, df['Volume'])  # volumen en barras
plt.subplot2grid((4, 4), (0, 0), rowspan=3, colspan=4)  # gráfico superior de una cuadrícula 4x4
(serie / serie.shift(1) - 1) * 100  # cambio porcentual diario
serie.pct_change() * 100  # lo mismo con el método de pandas
plt.hist(cambios, bins=50)  # histograma con 50 tramos
cambios.cumsum()  # rendimiento acumulado simple
((1 + r).cumprod() - 1) * 100  # rendimiento acumulado compuesto (r en fracción)
cambios.corr()  # matriz de correlación
plt.imshow(df_corr, cmap='hot', interpolation='none')  # mapa de calor de la correlación
cambios.rolling(window=75).std()  # volatilidad con ventana móvil de 75 días
```

**Errores comunes**

- Calcular la correlación sobre los precios en lugar de sobre los cambios diarios: dos series que suben con el tiempo parecen relacionadas sin estarlo.
- Creer que `.cumsum()` de los porcentajes diarios da el rendimiento real; solo es una aproximación simple, el real es compuesto.
- Dividir sin desplazar la serie, o usar `shift(-1)`, que compara con el día siguiente.
- Escribir `pivot(index=None, ...)` como el PDF: en pandas 3.1 lanza `KeyError`; se debe omitir `index`.
- Usar `end=datetime.today()` y esperar resultados repetibles: el rango cambia cada día, y `end` queda excluido.
- Llamar `plt.legend()` una sola vez con varios ejes: solo afecta al eje activo.

## Unidad 40 — Miniproyecto de análisis de la pandemia de Coronavirus en el mundo

[Ir a la unidad](clase-2-mini-proyectos/u40-pandemia-coronavirus.ipynb)

```python
pip install folium  # instala la biblioteca
import folium  # importa Folium
m = folium.Map(location=[lat, lon], zoom_start=13)  # crea un mapa centrado en una coordenada
m = folium.Map(location=[lat, lon], zoom_start=13, tiles="cartodbpositron")  # cambia el estilo del mapa base
folium.Marker(location=[lat, lon], popup="Texto", icon=folium.Icon(color="green")).add_to(m)  # marcador con mensaje emergente
folium.CircleMarker(location=[lat, lon], radius=100, popup="circle", color="#3186cc", fill=True, fill_color="#3186cc").add_to(m)  # círculo
folium.Choropleth(geo_data=geo, data=tabla, columns=("Code", "valor"), key_on="feature.id", fill_color="YlGn").add_to(m)  # mapa coroplético
folium.LayerControl().add_to(m)  # control para activar o desactivar capas
m.save("index.html")  # guarda el mapa como archivo HTML
m  # última expresión de la celda: muestra el mapa
df["Date"] = pd.to_datetime(df["Day"])  # convierte el texto en fecha
df = df.set_index("Date").drop(["Day"], axis=1)  # fecha como índice, sin la columna original
df.sort_index().groupby("Entity").last()  # último valor acumulado de cada entidad
geo = json.load(open(ruta, encoding="utf-8"))  # carga un archivo GeoJSON
```

**Errores comunes**

- Sumar con `sum()` una columna que ya es un acumulado: usar el último valor (`last()`) o el máximo.
- Mezclar países con grupos agregados (África, Asia, Mundo) al ordenar o calcular estadísticas.
- Invertir latitud y longitud al crear el mapa o los marcadores.
- Que `key_on` no coincida con la columna identificadora (por ejemplo nombres frente a códigos): las áreas quedan sin color.
- Nombrar `map` al objeto mapa y ocultar la función incorporada `map()`.
- Usar `inplace=True` en pandas 3: está en desuso; se recomienda reasignar el resultado.

[Índice de la semana](semana-06.ipynb)